# Classifier test (qwen3 14b / 8b / 4b / 1.7b)

Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

- **Exact**: predicted domains == gold domains
- **Covers gold**: no gold domain missing (extras allowed)

In [ ]:
# Install and start Ollama, then pull the models
import subprocess, time, requests

CLASSIFIERS = ["qwen3:14b", "qwen3:8b", "qwen3:4b-q4_K_M", "qwen3:1.7b"]

subprocess.run("apt-get install -y -qq zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh",
               shell=True, check=True)
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
for m in CLASSIFIERS:
    print("pulling", m)
    subprocess.run(["ollama", "pull", m], check=True, stdout=subprocess.DEVNULL)
print("ready")


In [ ]:
# Classifier prompt and test rows, copied from prompts.py and sft_test.jsonl
CLASSIFIER_SYSTEM = 'You are a logic puzzle domain classifier. Your only job is to identify which domains a puzzle uses. Do NOT attempt to solve it.\n\nAvailable domains:\n  ordering           - entities occupy ordered positions or slots (first, last, before, after)\n  knights_and_knaves - characters make statements; some always lie, some always tell the truth\n  grouping           - entities are assigned to categories or groups\n\nReason through which domains the puzzle requires, then output your final answer on the last line in exactly this format:\nACTIVE_DOMAINS: <comma-separated, no spaces>\n\nExample: ACTIVE_DOMAINS: ordering,knights_and_knaves'

DOMAINS = ['ordering', 'knights_and_knaves', 'grouping']

ROWS = [
 {
  "problem_text": "There are 3 positions (numbered 1 through 3). Every person is either a knight (always tells the truth) or a knave (always lies). Pamela is in position 3. Amy is before Olga. Olga is a knight. If Amy is before Olga, then Olga is a knight. If Amy is a knight, then Pamela and Amy are adjacent. If Amy is a knave, then it is not the case that: Pamela and Amy are adjacent. If Amy is a knight, then Olga is immediately before Amy. If Amy is a knave, then it is not the case that: Olga is immediately before Amy. Which of the following is correct?\nA) [could be false] Pamela is a knave\nB) [must be false] Pamela is a knight",
  "active_domains": [
   "knights_and_knaves",
   "ordering"
  ]
 },
 {
  "problem_text": "There are 6 positions numbered 1 through 6. Thomas, Louise, Ana, Samuel, Orla, and Sage are assigned to these positions. Thomas is before Ana. Orla is before Thomas. Orla is before Ana. Ana is immediately before Louise. Sage is immediately before Samuel. Orla and Sage are adjacent. Which of the following is correct?\nA) [could be true] Thomas is in slot 4\nB) [could be true] Thomas is in slot 6",
  "active_domains": [
   "ordering"
  ]
 },
 {
  "problem_text": "There are 4 positions numbered 1 through 4. Ivan, George, Lois, and Julie occupy these positions. Ivan is in slot 1. Lois is in slot 2. Ivan is before Lois. Lois is before George. Lois is immediately before George. Given that Ivan is before Julie, which of the following is correct?\nA) [must be true] Ivan is in slot 1\nB) [must be true] Ivan is in slot 4\nC) [must be true] Ivan is in slot 2\nD) [could be true] Ivan is in slot 3",
  "active_domains": [
   "ordering"
  ]
 },
 {
  "problem_text": "There are 4 positions (numbered 1 through 4). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Joyce, Troy, Tina, and Hugo are the people in these positions. Joyce is in slot 2. Hugo is before Tina. Joyce is in group 2. If Joyce and Hugo are adjacent, then exactly 1 of [Joyce, Troy, Tina, Hugo] are in group 1. If Hugo is a truth-teller (knight), then Troy is before Hugo. If Hugo is a deceiver (knave), then it is not the case that: Troy is before Hugo. If Tina is a truth-teller (knight), then Joyce is immediately before Hugo. If Tina is a deceiver (knave), then it is not the case that: Joyce is immediately before Hugo. Given that exactly 1 of [Troy, Tina, Hugo] are in group 1, which of the following is correct?\nA) [must be true] Joyce is in slot 2\nB) [could be true] Joyce is in slot 4\nC) [must be true] Joyce is in slot 1",
  "active_domains": [
   "ordering",
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 6 positions (numbered 1 through 6). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Exactly 2 of [Joseph, Alton, Omar] are in group 2. Exactly 1 of [Casey, Alton, Betty] are in group 1. If Alton is before Joseph, then exactly 2 of [Joseph, Alton, Omar] are in group 2. If Betty is a deceiver (knave), then exactly 3 of [Maple, Casey, Alton, Omar] are in group 2. If Alton is a truth-teller (knight), then Alton is in slot 2. If Alton is a deceiver (knave), then it is not the case that: Alton is in slot 2. If Maple is a truth-teller (knight), then Omar is immediately before Alton. If Maple is a deceiver (knave), then it is not the case that: Omar is immediately before Alton. Which of the following is correct?\nA) [could be false] Joseph is a deceiver (knave)\nB) [must be false] Joseph is a truth-teller (knight)",
  "active_domains": [
   "knights_and_knaves",
   "ordering",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 6 positions (numbered 1 through 6). Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Dave is before Ivan. If Dara is in slot 6, then exactly 2 of [Judy, Ivan, Sarah] are in group 3. If Dave is before Maryann, then Dara and Dave are in different groups. If Judy is in slot 4, then exactly 1 of [Dara, Dave, Sarah] are in group 3. If Dave is a truth-teller (knight), then exactly 2 of [Judy, Maryann, Sarah] are in group 1. If Dave is a deceiver (knave), then it is not the case that: exactly 2 of [Judy, Maryann, Sarah] are in group 1. If Dara is a truth-teller (knight), then exactly 1 of [Judy, Dara, Ivan] are in group 2. If Dara is a deceiver (knave), then it is not the case that: exactly 1 of [Judy, Dara, Ivan] are in group 2. Which of the following is correct?\nA) [must be false] Ivan is a deceiver (knave)\nB) [could be false] Ivan is a truth-teller (knight)",
  "active_domains": [
   "knights_and_knaves",
   "ordering",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 2 groups. Carolyn, Rodger, Wade, and Grace are in these groups. Carolyn and Wade are in different groups. Exactly 2 of Carolyn, Rodger, Wade are in group 1. Exactly 1 of Carolyn, Rodger, Grace are in group 2. Which of the following is correct?\nA) [must be false] Wade is in group 2\nB) [could be false] Wade is in group 1",
  "active_domains": [
   "grouping"
  ]
 },
 {
  "problem_text": "There are 2 positions numbered 1 through 2. Every person is either a knight or a knave. There are 2 groups. Rodger is before Dara. Rodger is a knight. Exactly 1 of Rodger and Dara are in group 1. If exactly 1 of Rodger and Dara are in group 2, then exactly 1 of Rodger and Dara are in group 1. If Dara is a knight, then Rodger is in group 2. If Dara is a knave, then it is not the case that: Rodger is in group 2. If Rodger is a knight, then Rodger and Dara are adjacent. If Rodger is a knave, then it is not the case that: Rodger and Dara are adjacent. Given that if exactly 1 of Rodger and Dara are in group 1, then Dara is in slot 2, which of the following is correct?\nA) [could be true] Dara is in slot 1\nB) [must be true] Dara is in slot 2",
  "active_domains": [
   "ordering",
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 2 positions numbered 1 through 2. Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. If Andy is immediately before Carson, then Carson is in slot 2. If Andy and Carson are adjacent, then exactly 1 of [Andy, Carson] are in group 1. If exactly 1 of [Andy, Carson] are in group 2, then Andy and Carson are adjacent. If Andy is before Carson, then Andy is in group 2. If Andy is a truth-teller (knight), then Andy and Carson are in the same group. If Andy is a deceiver (knave), then it is not the case that: Andy and Carson are in the same group. If Andy is a truth-teller (knight), then Carson is in group 2. If Andy is a deceiver (knave), then it is not the case that: Carson is in group 2. Given that Andy and Carson are adjacent, which of the following is correct?\nA) [must be true] Andy is in group 2\nB) [must be true] Andy is in group 1",
  "active_domains": [
   "ordering",
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 5 positions (numbered 1 through 5). Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Jan is in group 3. exactly 1 of [Daniel, Maria, Heidi] are in group 2. exactly 1 of [Anthony, Maria, Jan] are in group 2. If Daniel is a truth-teller (knight), then Heidi is in group 1. If Anthony is a truth-teller (knight), then Maria and Heidi are in the same group. If Anthony is a deceiver (knave), then it is not the case that: Maria and Heidi are in the same group. If Heidi is a truth-teller (knight), then Daniel and Anthony are adjacent. If Heidi is a deceiver (knave), then it is not the case that: Daniel and Anthony are adjacent. Which of the following is correct?\nA) [must be true] Daniel is in group 2\nB) [could be false] Daniel is in group 1",
  "active_domains": [
   "grouping",
   "ordering",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 3 positions numbered 1 through 3. Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Luna is in group 3. exactly 1 of [Luna, Edgar, Edward] are in group 2. exactly 1 of [Luna, Edgar, Edward] are in group 1. exactly 1 of [Luna, Edgar, Edward] are in group 3. If Luna is in slot 3, then exactly 1 of [Luna, Edgar, Edward] are in group 2. If Luna is a truth-teller (knight), then Luna is in slot 1. If Luna is a deceiver (knave), then it is not the case that: Luna is in slot 1. If Luna is a truth-teller (knight), then Edgar is immediately before Luna. If Luna is a deceiver (knave), then it is not the case that: Edgar is immediately before Luna. Which of the following is correct?\nA) [must be true] Edward is a deceiver (knave)\nB) [could be false] Edward is a truth-teller (knight)",
  "active_domains": [
   "ordering",
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 5 positions (numbered 1 through 5). Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Exactly 1 of [Brandi, Mildred, Richard] are in group 2. Exactly 1 of [Brandi, Mildred, Richard, Kristofer] are in group 2. If exactly 2 of [Mildred, Richard, Sage] are in group 1, then Mildred and Sage are adjacent. If Kristofer is a truth-teller (knight), then Mildred is in group 2. If Kristofer is a deceiver (knave), then it is not the case that: Mildred is in group 2. If Mildred is a truth-teller (knight), then exactly 1 of [Brandi, Mildred, Kristofer] are in group 3. If Mildred is a deceiver (knave), then it is not the case that: exactly 1 of [Brandi, Mildred, Kristofer] are in group 3. Which of the following is correct?\nA) [could be true] Mildred is in group 1\nB) [must be false] Mildred is in group 3\nC) [must be false] Mildred is in group 2",
  "active_domains": [
   "ordering",
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Bonnie is a deceiver (knave). Knox is in group 2. exactly 1 of [Bonnie, Knox] are in group 1. exactly 1 of [Bonnie, Knox] are in group 2. If Knox is a truth-teller (knight), then Knox is in group 2. If Knox is in group 2, then Knox is a truth-teller (knight). If Bonnie is a truth-teller (knight), then Knox is a deceiver (knave). If Bonnie is a deceiver (knave), then it is not the case that: Knox is a deceiver (knave). If Knox is a truth-teller (knight), then exactly 1 of [Bonnie, Knox] are in group 1. If Knox is a deceiver (knave), then it is not the case that: exactly 1 of [Bonnie, Knox] are in group 1. Which of the following is correct?\nA) [could be true] Bonnie is a deceiver (knave)\nB) [must be true] Bonnie is a truth-teller (knight)",
  "active_domains": [
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 3 positions (numbered 1 through 3). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Carol is in slot 2. Jorge is before Hugo. Jorge and Carol are in the same group. exactly 2 of [Jorge, Carol, Hugo] are in group 1. If exactly 1 of [Jorge, Carol, Hugo] are in group 2, then Jorge is immediately before Carol. If Hugo is a truth-teller (knight), then Hugo is in group 1. If Hugo is a deceiver (knave), then it is not the case that: Hugo is in group 1. If Jorge is a truth-teller (knight), then Jorge and Hugo are in different groups. If Jorge is a deceiver (knave), then it is not the case that: Jorge and Hugo are in different groups. Given that Jorge and Hugo are in different groups, which of the following is correct?\nA) [could be true] Hugo is a truth-teller (knight)\nB) [could be true] Hugo is a deceiver (knave)",
  "active_domains": [
   "ordering",
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 4 positions numbered 1 through 4. Every person is either a knight or a knave. Katie is in slot 3. Vernon is in slot 2. Vernon is before Katie. Katie is a knight. It is not the case that Eric is in slot 3. If Vernon is a knight, then Eric is immediately before Katie. If Vernon is a knave, then it is not the case that Eric is immediately before Katie. If Eric is a knight, then Katie and Nina are adjacent. If Eric is a knave, then it is not the case that Katie and Nina are adjacent. Which of the following is correct?\nA) [could be true] Katie is in slot 3\nB) [must be true] Katie is in slot 4\nC) [could be true] Katie is in slot 2",
  "active_domains": [
   "knights_and_knaves",
   "ordering"
  ]
 },
 {
  "problem_text": "There are 5 positions numbered 1 through 5. Every person is either a knight who always tells the truth or a knave who always lies. There are 2 groups. Finn, Linda, Bob, Theresa, and Wendy are the individuals involved. Bob is in position 3. Bob is before Finn. It is not the case that Wendy is in position 5. If Finn is a knight, then Bob and Wendy are adjacent. If Finn is a knave, then it is not the case that Bob and Wendy are adjacent. If Linda is a knight, then Finn is immediately before Theresa. If Linda is a knave, then it is not the case that Finn is immediately before Theresa. Given that Bob is immediately before Theresa, which of the following is correct?\nA) [could be true] Wendy is a knave\nB) [must be false] Wendy is a knight",
  "active_domains": [
   "knights_and_knaves",
   "ordering",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 2 positions numbered 1 through 2. Every person is either a knight or a knave. There are 2 groups. Liam is in slot 2. Liam and Elinor are adjacent. If Liam is in group 1, then Liam and Elinor are adjacent. If Liam is in slot 2, then Liam is in group 1. If Liam is a knight, then Liam is in slot 2. If Liam is a knave, then it is not the case that: Liam is in slot 2. If Elinor is a knight, then Liam and Elinor are in the same group. If Elinor is a knave, then it is not the case that: Liam and Elinor are in the same group. Which of the following is correct?\nA) [could be true] Elinor is in slot 1\nB) [could be true] Elinor is in slot 2",
  "active_domains": [
   "grouping",
   "knights_and_knaves",
   "ordering"
  ]
 },
 {
  "problem_text": "There are 5 positions, numbered 1 through 5. Every person is either a knight, who always tells the truth, or a knave, who always lies. Donald, Lena, Mary, Gretchen, and Juno occupy these positions.\nLena is before Juno.\nMary is immediately before Juno.\nIf Gretchen is immediately before Donald, then Lena is a knight.\nIf Lena is a knight, then Mary and Juno are adjacent.\nIf Lena is a knave, then it is not the case that Mary and Juno are adjacent.\nIf Juno is a knight, then Donald is immediately before Lena.\nIf Juno is a knave, then it is not the case that Donald is immediately before Lena.\nWhich of the following is correct?\nA) [must be false] Donald is in slot 4\nB) [could be false] Donald is in slot 5\nC) [must be false] Donald is in slot 1",
  "active_domains": [
   "ordering",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 4 positions numbered 1 through 4. Every person is either a knight who always tells the truth or a knave who always lies. There are 3 groups. Exactly 2 of Ronald, Lena, and Tara are in group 3. Exactly 1 of Donald, Lena, and Tara are in group 2. If Donald is in slot 1, then Ronald and Tara are in different groups. If Donald is a knight, then Lena is immediately before Tara. If Donald is a knave, then it is not the case that: Lena is immediately before Tara. If Donald is a knight, then Donald is in slot 2. If Donald is a knave, then it is not the case that: Donald is in slot 2. Given that Donald and Lena are adjacent, which of the following is correct?\nA) [must be true] Tara is in slot 3\nB) [could be true] Tara is in slot 4\nC) [must be true] Tara is in slot 2\nD) [must be false] Tara is in slot 1",
  "active_domains": [
   "knights_and_knaves",
   "grouping",
   "ordering"
  ]
 },
 {
  "problem_text": "There are six positions, numbered 1 through 6. The people involved are Susan, Kathryn, Miguel, Ramon, James, and Jeremy.\nKathryn is in slot 6.\nRamon is before Susan.\nSusan is immediately before Kathryn.\nSusan and Kathryn are adjacent.\nWhich of the following is correct?\nA) [must be true] Ramon is in slot 1\nB) [could be true] Ramon is in slot 3\nC) [must be true] Ramon is in slot 6",
  "active_domains": [
   "ordering"
  ]
 },
 {
  "problem_text": "There are 2 groups. Maple, Jessica, Myra, and Joan are each in one of these two groups. Jessica and Myra are in different groups. Jessica and Joan are in the same group. exactly 2 of [Maple, Myra, Joan] are in group 1. exactly 1 of [Maple, Myra, Joan] are in group 2. exactly 2 of [Jessica, Myra, Joan] are in group 1. it is not the case that: Maple and Myra are in the same group. If Jessica and Joan are in the same group, then exactly 3 of [Maple, Jessica, Myra, Joan] are in group 1. Which of the following is correct?\nA) [must be true] Joan is in group 2\nB) [must be true] Joan is in group 1",
  "active_domains": [
   "grouping"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). Michelle, Dara, and Debra are the persons under consideration.\nMichelle is a deceiver (knave).\nDara is a deceiver (knave).\nDebra is a truth-teller (knight).\nIf Michelle is a truth-teller (knight), then Debra is a deceiver (knave).\nIf Michelle is a deceiver (knave), then it is not the case that: Debra is a deceiver (knave).\nIf Debra is a truth-teller (knight), then Michelle is a deceiver (knave).\nIf Debra is a deceiver (knave), then it is not the case that: Michelle is a deceiver (knave).\nWhich of the following is correct?\nA) [could be true] Dara is a truth-teller (knight)\nB) [could be true] Dara is a deceiver (knave)",
  "active_domains": [
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 2 positions (numbered 1 through 2). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Brandi is in group 2. If Brandi is a deceiver (knave), then Brandi and Karl are in different groups. If exactly 1 of [Brandi, Karl] are in group 1, then Karl is a deceiver (knave). If Brandi is a truth-teller (knight), then Brandi and Karl are in the same group. If Brandi is a deceiver (knave), then it is not the case that: Brandi and Karl are in the same group. If Karl is a truth-teller (knight), then Brandi is in slot 2. If Karl is a deceiver (knave), then it is not the case that: Brandi is in slot 2. Which of the following is correct?\nA) [could be false] Karl is a deceiver (knave)\nB) [must be false] Karl is a truth-teller (knight)",
  "active_domains": [
   "ordering",
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). Dara is a truth-teller (knight). Judith is a deceiver (knave). If Judith is a truth-teller (knight), then Dara is a deceiver (knave). If Judith is a deceiver (knave), then it is not the case that: Dara is a deceiver (knave). If Dara is a truth-teller (knight), then Judith is a deceiver (knave). If Dara is a deceiver (knave), then it is not the case that: Judith is a deceiver (knave). Which of the following is correct?\nA) [must be true] Judith is a deceiver (knave)\nB) [could be true] Judith is a truth-teller (knight)",
  "active_domains": [
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). Liam is a deceiver (knave). Penn is a deceiver (knave). If Penn is a truth-teller (knight), then Liam is a truth-teller (knight). If Penn is a deceiver (knave), then it is not the case that: Liam is a truth-teller (knight). If Liam is a truth-teller (knight), then Penn is a truth-teller (knight). If Liam is a deceiver (knave), then it is not the case that: Penn is a truth-teller (knight). Which of the following is correct?\nA) [could be true] Liam is a deceiver (knave)\nB) [must be true] Liam is a truth-teller (knight)",
  "active_domains": [
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 6 positions (numbered 1 through 6). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Hollis is in slot 6. Exactly 1 of [Lisa, June, Ursula] are in group 1. If exactly 1 of [Alice, Lisa, June] are in group 2, then exactly 2 of [Alice, Lisa, June, Ursula] are in group 2. If Alice and Ursula are adjacent, then exactly 2 of [Alice, June, Annette] are in group 1. If Ursula is a truth-teller (knight), then exactly 2 of [Alice, June, Ursula] are in group 2. If Ursula is a deceiver (knave), then it is not the case that: exactly 2 of [Alice, June, Ursula] are in group 2. If Alice is a truth-teller (knight), then Annette is before Ursula. If Alice is a deceiver (knave), then it is not the case that: Annette is before Ursula. Which of the following is correct?\nA) [could be true] Annette is in slot 1\nB) [must be false] Annette is in slot 5\nC) [must be true] Annette is in slot 3",
  "active_domains": [
   "knights_and_knaves",
   "ordering",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 5 positions (numbered 1 through 5). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Roxanne, Andy, Theo, Dong, and Mary are the people involved. Mary is before Dong. exactly 2 of [Roxanne, Theo, Dong] are in group 1. If Andy and Mary are adjacent, then exactly 2 of [Roxanne, Theo, Dong, Mary] are in group 1. If Theo and Dong are adjacent, then Andy is a truth-teller (knight). If Mary is a truth-teller (knight), then Dong is a deceiver (knave). If Mary is a deceiver (knave), then it is not the case that: Dong is a deceiver (knave). If Roxanne is a truth-teller (knight), then Theo is a deceiver (knave). If Roxanne is a deceiver (knave), then it is not the case that: Theo is a deceiver (knave). Which of the following is correct?\nA) [must be false] Mary is in group 1\nB) [could be false] Mary is in group 2",
  "active_domains": [
   "ordering",
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). Sarah, Marianne, Jesus, Sage, and Shayla are such people. Sarah is a truth-teller (knight). Marianne is a truth-teller (knight). Jesus is a deceiver (knave). Sage is a truth-teller (knight). Shayla is a deceiver (knave). If Marianne is a truth-teller (knight), then Sage is a truth-teller (knight). If Marianne is a deceiver (knave), then it is not the case that: Sage is a truth-teller (knight). If Sage is a truth-teller (knight), then Marianne is a truth-teller (knight). If Sage is a deceiver (knave), then it is not the case that: Marianne is a truth-teller (knight). Which of the following is correct?\nA) [could be true] Sage is a truth-teller (knight)\nB) [must be true] Sage is a deceiver (knave)",
  "active_domains": [
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 5 positions (numbered 1 through 5). Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Theo is in slot 4. Reed and Elinor are in different groups. exactly 1 of [Jacqueline, Theo, Elinor, Luna] are in group 1. If Theo is in slot 4, then Elinor is before Theo. If exactly 1 of [Jacqueline, Theo, Elinor, Luna] are in group 3, then Elinor is before Theo. If exactly 1 of [Jacqueline, Theo, Reed, Elinor] are in group 3, then Elinor is before Jacqueline. If Jacqueline is a truth-teller (knight), then exactly 1 of [Jacqueline, Reed, Elinor] are in group 1. If Jacqueline is a deceiver (knave), then it is not the case that: exactly 1 of [Jacqueline, Reed, Elinor] are in group 1. If Theo is a truth-teller (knight), then Jacqueline is before Theo. If Theo is a deceiver (knave), then it is not the case that: Jacqueline is before Theo. Which of the following is correct?\nA) [could be true] Reed is a truth-teller (knight)\nB) [must be false] Reed is a deceiver (knave)",
  "active_domains": [
   "grouping",
   "knights_and_knaves",
   "ordering"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. John is a knave. Cyrus is a knight. Kathryn and Maria are in different groups. exactly 1 of [John, Kathryn, Cyrus] are in group 2. exactly 1 of [John, Cyrus, Maria] are in group 1. If Cyrus is a knight, then exactly 1 of [John, Cyrus, Maria] are in group 3. If Cyrus is a knight, then Cyrus is in group 1. If Cyrus is a knave, then it is not the case that: Cyrus is in group 1. If Cyrus is a knight, then exactly 1 of [John, Kathryn, Cyrus, Maria] are in group 3. If Cyrus is a knave, then it is not the case that: exactly 1 of [John, Kathryn, Cyrus, Maria] are in group 3. Which of the following is correct?\nA) [could be false] Kathryn is a knave\nB) [must be false] Kathryn is a knight",
  "active_domains": [
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 4 positions numbered 1 through 4. Nova, Maurice, Morgan, and David are assigned to these positions. Morgan is in slot 1. Morgan is before Nova. David is before Nova. David is before Maurice. Maurice is immediately before Nova. Morgan is immediately before David. Morgan and David are adjacent. Which of the following is correct?\nA) [could be true] Morgan is in slot 4\nB) [could be true] Morgan is in slot 3\nC) [must be true] Morgan is in slot 1\nD) [could be true] Morgan is in slot 2",
  "active_domains": [
   "ordering"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Theo is in group 2. Exactly 1 of Aria, Theo, and Brad are in group 3. Exactly 1 of Aria, Victor, and Brad are in group 1. Exactly 2 of Aria, Theo, Victor, and Brad are in group 2. If Victor is a knight, then Aria and Victor are in different groups. If Victor is a knave, then it is not the case that: Aria and Victor are in different groups. If Theo is a knight, then Victor and Brad are in the same group. If Theo is a knave, then it is not the case that: Victor and Brad are in the same group. Given that if Victor is a knight, then exactly 2 of Aria, Theo, Victor, and Brad are in group 2, which of the following is correct?\nA) [must be true] Theo is in group 3\nB) [must be true] Theo is in group 2",
  "active_domains": [
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 5 positions (numbered 1 through 5). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Heidi and Liam are in different groups. exactly 1 of [Heidi, Luna, Liam] are in group 1. exactly 3 of [Zara, Heidi, Luna, Liam] are in group 2. If Luna is before Zara, then Blanca is in group 1. If Liam is before Zara, then Zara and Heidi are in different groups. If Blanca is a truth-teller (knight), then Blanca is in slot 1. If Blanca is a deceiver (knave), then it is not the case that: Blanca is in slot 1. If Luna is a truth-teller (knight), then Blanca is immediately before Liam. If Luna is a deceiver (knave), then it is not the case that: Blanca is immediately before Liam. Which of the following is correct?\nA) [could be false] Heidi is in group 1\nB) [must be false] Heidi is in group 2",
  "active_domains": [
   "ordering",
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 3 groups. Leslie, Eugene, Mona, Eleanor, Michael, and Norman are each assigned to one of these three groups. Eleanor is in group 3. exactly 2 of [Eugene, Eleanor, Norman] are in group 3. exactly 3 of [Mona, Eleanor, Michael, Norman] are in group 3. Which of the following is correct?\nA) [must be false] Eugene is in group 3\nB) [could be false] Eugene is in group 2",
  "active_domains": [
   "grouping"
  ]
 },
 {
  "problem_text": "There are 6 positions (numbered 1 through 6). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Katherine, Vilma, Dave, Christine, William, and Brian are assigned to these positions and groups. Christine is before William. Exactly 1 of [Vilma, Dave, Christine] are in group 2. Exactly 1 of [Dave, Christine, Brian] are in group 2. If Brian is a truth-teller (knight), then exactly 2 of [Katherine, Vilma, Christine, William] are in group 1. If Brian is a deceiver (knave), then it is not the case that: exactly 2 of [Katherine, Vilma, Christine, William] are in group 1. If Dave is a truth-teller (knight), then exactly 1 of [Katherine, William, Brian] are in group 2. If Dave is a deceiver (knave), then it is not the case that: exactly 1 of [Katherine, William, Brian] are in group 2. Which of the following is correct?\nA) [must be true] William is in slot 2\nB) [could be false] William is in slot 4",
  "active_domains": [
   "ordering",
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Nova and Theo are in different groups. It is not the case that: Kristofer is a deceiver (knave). If John and Kristofer are in different groups, then Nova and Theo are in different groups. If exactly 1 of [Nova, Theo, Kristofer] are in group 1, then Theo is a truth-teller (knight). If Nova is a truth-teller (knight), then exactly 2 of [Nova, Theo, John] are in group 1. If Nova is a deceiver (knave), then it is not the case that: exactly 2 of [Nova, Theo, John] are in group 1. If Theo is a truth-teller (knight), then exactly 2 of [Nova, Theo, John, Kristofer] are in group 1. If Theo is a deceiver (knave), then it is not the case that: exactly 2 of [Nova, Theo, John, Kristofer] are in group 1. Which of the following is correct?\nA) [could be false] Theo is in group 1\nB) [must be false] Theo is in group 2",
  "active_domains": [
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 3 positions (numbered 1 through 3). Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Luther is immediately before Victor. Uma is a truth-teller (knight). Luther is a deceiver (knave). Victor is a deceiver (knave). Uma and Luther are in different groups. Victor is in group 3. exactly 1 of [Uma, Luther, Victor] are in group 3. If Luther and Victor are in different groups, then Victor is in slot 2. If Victor is in slot 2, then Luther is in group 2. If Luther is a truth-teller (knight), then Uma is before Victor. If Luther is a deceiver (knave), then it is not the case that: Uma is before Victor. If Uma is a truth-teller (knight), then Luther is in group 2. If Uma is a deceiver (knave), then it is not the case that: Luther is in group 2. Which of the following is correct?\nA) [must be true] Luther is in slot 2\nB) [must be true] Luther is in slot 3\nC) [must be true] Luther is in slot 1",
  "active_domains": [
   "ordering",
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 5 positions numbered 1 through 5. Every person is either a knight or a knave, and there are 2 groups. David is in position 3. Donna and Victor are in different groups. Exactly 3 of [Theo, Chung, Donna, Victor] are in group 2. If Chung is a knight, then Donna is a knight. If Chung is a knave, then it is not the case that: Donna is a knight. If Victor is a knight, then David and Donna are in different groups. If Victor is a knave, then it is not the case that: David and Donna are in different groups. Which of the following is correct?\nA) [could be true] Chung is in position 3\nB) [could be false] Chung is in position 5",
  "active_domains": [
   "grouping",
   "ordering",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 3 positions (numbered 1 through 3). Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Mona is in slot 1. Daniel is in slot 3. Mona and Daniel are in different groups. If exactly 1 of Mona, Daniel, Luna are in group 1, then Luna is a deceiver (knave). If Mona and Luna are adjacent, then Daniel and Luna are in different groups. If Luna is a truth-teller (knight), then Luna is in group 2. If Luna is a deceiver (knave), then it is not the case that: Luna is in group 2. If Luna is a truth-teller (knight), then Daniel and Luna are in the same group. If Luna is a deceiver (knave), then it is not the case that: Daniel and Luna are in the same group. Given that Mona and Luna are in different groups, which of the following is correct?\nA) [could be false] Mona is a truth-teller (knight)\nB) [must be false] Mona is a deceiver (knave)",
  "active_domains": [
   "ordering",
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 2 positions (numbered 1 through 2). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Reed is before Michael. Michael and Reed are adjacent. Michael is a truth-teller (knight). exactly 1 of [Michael, Reed] are in group 2. If Michael is in slot 2, then exactly 1 of [Michael, Reed] are in group 1. If Reed is a truth-teller (knight), then Michael and Reed are adjacent. If Reed is a deceiver (knave), then it is not the case that: Michael and Reed are adjacent. If Michael is a truth-teller (knight), then Reed is in slot 1. If Michael is a deceiver (knave), then it is not the case that: Reed is in slot 1. Which of the following is correct?\nA) [must be true] Reed is a deceiver (knave)\nB) [could be true] Reed is a truth-teller (knight)",
  "active_domains": [
   "ordering",
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 3 positions numbered 1 through 3. The entities involved are Tina, Richard, and Marian. Marian is in position 2. Marian is before Tina. Richard is before Marian. Marian is immediately before Tina. Tina and Marian are adjacent. If Richard is immediately before Marian, then Richard is in position 1. Which of the following is correct?\nA) [must be true] Tina is in position 2\nB) [could be true] Tina is in position 3\nC) [must be true] Tina is in position 1",
  "active_domains": [
   "ordering"
  ]
 },
 {
  "problem_text": "There are 6 positions numbered 1 through 6 for Gordon, Karl, Joseph, Eve, Christopher, and Dara. Joseph is in slot 3. Eve is in slot 1. If Gordon is in slot 2, then Christopher is immediately before Karl. Which of the following is correct?\nA) [must be true] Joseph is in slot 2\nB) [could be true] Joseph is in slot 3\nC) [could be true] Joseph is in slot 4\nD) [must be true] Joseph is in slot 1",
  "active_domains": [
   "ordering"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Zane and Reed are in different groups. Exactly 2 of [Reed, Tristan, Thomas] are in group 3. If exactly 1 of [Zane, Reed, Tristan, Thomas] are in group 1, then Zane is a truth-teller (knight). If exactly 1 of [Zane, Reed, Tristan] are in group 1, then Reed is a deceiver (knave). If Thomas is a truth-teller (knight), then exactly 1 of [Alvin, Zane, Tristan] are in group 2. If exactly 1 of [Alvin, Reed, Tristan] are in group 2, then Thomas is a truth-teller (knight). If Reed is a truth-teller (knight), then Alvin is in group 2. If Reed is a deceiver (knave), then it is not the case that: Alvin is in group 2. If Thomas is a truth-teller (knight), then exactly 1 of [Alvin, Zane, Tristan] are in group 2. If Thomas is a deceiver (knave), then it is not the case that: exactly 1 of [Alvin, Zane, Tristan] are in group 2. Given that Alvin is in group 3, which of the following is correct?\nA) [must be true] Alvin is a truth-teller (knight)\nB) [could be false] Alvin is a deceiver (knave)",
  "active_domains": [
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 4 positions (numbered 1 through 4). There are 3 groups. Andy is in group 3. Exactly 1 of [Juan, Andy, Michelle] are in group 2. If Juan is immediately before Andy, then exactly 1 of [Lisa, Juan, Michelle] are in group 1. Which of the following is correct?\nA) [must be false] Lisa is in group 2\nB) [could be true] Lisa is in group 1\nC) [must be false] Lisa is in group 3",
  "active_domains": [
   "ordering",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 2 positions numbered 1 through 2. There are 2 groups. Uma is in slot 1. Reed is in group 1. Uma is in group 2. exactly 1 of [Reed, Uma] are in group 1. If Reed and Uma are in different groups, then Uma is immediately before Reed. If Reed is in group 1, then Uma is before Reed. If Uma is immediately before Reed, then Uma is in group 2. Which of the following is correct?\nA) [could be true] Uma is in slot 1\nB) [could be true] Uma is in slot 2",
  "active_domains": [
   "grouping",
   "ordering"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Brenda is a deceiver (knave). Noah is a deceiver (knave). Robert is a deceiver (knave). Exactly 2 of [Brenda, Robert, Priya] are in group 2. If Robert is in group 2, then Priya is a deceiver (knave). If Noah is a deceiver (knave), then exactly 2 of [Brenda, Noah, Priya] are in group 2. If Noah is a deceiver (knave), then Brenda and Noah are in the same group. If Robert is a truth-teller (knight), then Noah is a truth-teller (knight). If Robert is a deceiver (knave), then it is not the case that: Noah is a truth-teller (knight). If Robert is a truth-teller (knight), then Brenda and Noah are in different groups. If Robert is a deceiver (knave), then it is not the case that: Brenda and Noah are in different groups. Which of the following is correct?\nA) [must be true] Robert is in group 2\nB) [must be true] Robert is in group 1",
  "active_domains": [
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Wade is a knight. Veronica is a knight. If Veronica is a knight, then exactly 1 of [Wade, Veronica] are in group 1. If Wade is a knight, then Veronica is in group 1. If exactly 1 of [Wade, Veronica] are in group 2, then Wade is a knight. If Veronica is a knight, then Veronica is in group 1. If Veronica is a knight, then Wade is a knight. If Veronica is a knave, then it is not the case that: Wade is a knight. If Wade is a knight, then Veronica is a knight. If Wade is a knave, then it is not the case that: Veronica is a knight. Which of the following is correct?\nA) [could be true] Veronica is in group 1\nB) [could be true] Veronica is in group 2",
  "active_domains": [
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 6 positions (numbered 1 through 6). Every person is either a knight (always tells the truth) or a knave (always lies). Donald is in slot 1. Fletcher is a deceiver (knave). If Fletcher is a deceiver (knave), then Martin is immediately before Wendy. If Gwen is a truth-teller (knight), then Donald is before Orla. If Gwen is a deceiver (knave), then it is not the case that: Donald is before Orla. If Fletcher is a truth-teller (knight), then Wendy is before Orla. If Fletcher is a deceiver (knave), then it is not the case that: Wendy is before Orla. Which of the following is correct?\nA) [must be true] Orla is a truth-teller (knight)\nB) [could be true] Orla is a deceiver (knave)",
  "active_domains": [
   "knights_and_knaves",
   "ordering"
  ]
 },
 {
  "problem_text": "There are 5 positions (numbered 1 through 5). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Nicholas is in group 1. Exactly 2 of [Quinn, Mark, Jerry, Heidi, Nicholas] are in group 1. Exactly 2 of [Quinn, Heidi, Nicholas] are in group 2. Exactly 1 of [Mark, Jerry, Heidi] are in group 1. Exactly 2 of [Mark, Jerry, Heidi] are in group 2. If Mark is a truth-teller (knight), then Mark is before Jerry. If Mark is a deceiver (knave), then it is not the case that: Mark is before Jerry. If Heidi is a truth-teller (knight), then Mark is a truth-teller (knight). If Heidi is a deceiver (knave), then it is not the case that: Mark is a truth-teller (knight). Given that exactly 3 of [Quinn, Mark, Jerry, Heidi] are in group 2, which of the following is correct?\nA) [must be true] Quinn is in group 2\nB) [must be true] Quinn is in group 1",
  "active_domains": [
   "knights_and_knaves",
   "ordering",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 4 positions (numbered 1 through 4). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Nova, Alma, Bob, and Judith are assigned to these positions and groups. Alma is in group 2. If Bob is in slot 4, then exactly 3 of [Nova, Alma, Bob, Judith] are in group 2. If Alma is in group 2, then Nova and Alma are adjacent. If Bob is a truth-teller (knight), then Nova and Judith are in different groups. If Bob is a deceiver (knave), then it is not the case that: Nova and Judith are in different groups. If Bob is a truth-teller (knight), then Alma is in group 2. If Bob is a deceiver (knave), then it is not the case that: Alma is in group 2. Which of the following is correct?\nA) [could be true] Bob is in group 2\nB) [must be false] Bob is in group 1",
  "active_domains": [
   "knights_and_knaves",
   "ordering",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 5 positions numbered 1 through 5. There are 3 groups. Elinor is immediately before Ernest. Nancy is immediately before Maurice. Nancy and Rosa are in different groups. Elinor is in group 1. If Nancy is immediately before Maurice, then exactly 1 of [Nancy, Ernest, Rosa] are in group 3. If Elinor is in group 1, then Nancy is before Elinor. Which of the following is correct?\nA) [must be false] Rosa is in slot 5\nB) [could be true] Rosa is in slot 1",
  "active_domains": [
   "grouping",
   "ordering"
  ]
 },
 {
  "problem_text": "There are 4 positions (numbered 1 through 4). There are 2 groups (group sizes are NOT given here; they may or may not appear in the clues). Michael is immediately before Tara. Michael and Tara are adjacent. Michael and Tara are in different groups. Jerome and Linda are in different groups. Michael is in group 2. exactly 1 of [Michael, Jerome, Linda] are in group 1. Given that If Linda is in group 1, then Michael and Tara are adjacent. Which of the following is correct? A) [must be true] Tara is in slot 1 B) [must be false] Tara is in slot 3 C) [could be true] Tara is in slot 4 D) [must be true] Tara is in slot 2",
  "active_domains": [
   "ordering",
   "grouping"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Hugo is a truth-teller (knight). Penn and Hugo are in different groups. Hugo is in group 1. exactly 1 of [Penn, Hugo] are in group 1. If exactly 1 of [Penn, Hugo] are in group 2, then Hugo is a truth-teller (knight). If Penn is a deceiver (knave), then Hugo is in group 1. If exactly 1 of [Penn, Hugo] are in group 2, then Penn is a deceiver (knave). If Penn is a truth-teller (knight), then Penn and Hugo are in the same group. If Penn is a deceiver (knave), then it is not the case that: Penn and Hugo are in the same group. If Hugo is a truth-teller (knight), then Penn and Hugo are in different groups. If Hugo is a deceiver (knave), then it is not the case that: Penn and Hugo are in different groups. Which of the following is correct?\nA) [must be true] Hugo is in group 1\nB) [must be true] Hugo is in group 2",
  "active_domains": [
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 3 groups. Mars, Michelle, Mark, Orla, and Carol are each assigned to exactly one of these groups. Mars and Michelle are in different groups. exactly 1 of [Mars, Michelle, Orla] are in group 3. exactly 2 of [Mars, Michelle, Mark, Orla] are in group 2. exactly 2 of [Mars, Mark, Orla, Carol] are in group 3. If exactly 1 of [Mars, Michelle, Mark, Orla] are in group 1, then Mars and Mark are in different groups. Given that Michelle and Carol are in different groups, which of the following is correct?\nA) [must be true] Mark is in group 1\nB) [could be true] Mark is in group 2\nC) [must be true] Mark is in group 3",
  "active_domains": [
   "grouping"
  ]
 },
 {
  "problem_text": "Norman, Eve, Penn, Joanne, Jeanette, and Tommie are assigned to 3 groups. Norman and Penn are in different groups. exactly 1 of [Norman, Jeanette, Tommie] are in group 3. exactly 2 of [Eve, Penn, Joanne] are in group 1. exactly 1 of [Eve, Penn, Tommie] are in group 2. exactly 3 of [Norman, Joanne, Jeanette, Tommie] are in group 1. Given that exactly 1 of [Norman, Eve, Jeanette, Tommie] are in group 3, which of the following is correct?\nA) [could be false] Norman is in group 1\nB) [must be true] Norman is in group 2\nC) [must be true] Norman is in group 3",
  "active_domains": [
   "grouping"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Exactly 1 of [Tara, Tammy] are in group 1. If Tammy is a deceiver (knave), then exactly 1 of [Tara, Tammy] are in group 2. If Tammy is a deceiver (knave), then Tammy is in group 2. If Tara is a truth-teller (knight), then Tara is in group 2. If Tara is a deceiver (knave), then it is not the case that: Tara is in group 2. If Tammy is a truth-teller (knight), then Tara is a truth-teller (knight). If Tammy is a deceiver (knave), then it is not the case that: Tara is a truth-teller (knight). Which of the following is correct?\nA) [could be false] Tara is a deceiver (knave)\nB) [must be false] Tara is a truth-teller (knight)",
  "active_domains": [
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 2 positions numbered 1 through 2. Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Leslie is immediately before Allen. Allen is in group 1. Exactly 1 of [Allen, Leslie] are in group 1. Exactly 1 of [Allen, Leslie] are in group 2. If Allen is in group 1, then Leslie is a deceiver (knave). If Allen and Leslie are in different groups, then Allen and Leslie are adjacent. If Allen is a truth-teller (knight), then Allen is in group 1. If Allen is a deceiver (knave), then it is not the case that: Allen is in group 1. If Leslie is a truth-teller (knight), then Allen is before Leslie. If Leslie is a deceiver (knave), then it is not the case that: Allen is before Leslie. Which of the following is correct?\nA) [must be true] Leslie is a truth-teller (knight)\nB) [could be true] Leslie is a deceiver (knave)",
  "active_domains": [
   "ordering",
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 2 groups. Rosetta and Brenda are in different groups. Exactly 1 of Rosetta, Brenda are in group 1. Exactly 1 of Rosetta, Brenda are in group 2. Given that Brenda is in group 2, which of the following is correct?\nA) [must be true] Rosetta is in group 1\nB) [must be true] Rosetta is in group 2",
  "active_domains": [
   "grouping"
  ]
 },
 {
  "problem_text": "There are 6 positions (numbered 1 through 6). Every person is either a knight (always tells the truth) or a knave (always lies). Sam, Judith, Keith, Dawn, Grace, and Juanita occupy these positions. If Grace is before Dawn, then Dawn is a knave. If Keith is before Judith, then Grace is a knight. If Dawn and Grace are adjacent, then Keith is a knight. If Judith is a knight, then Keith is immediately before Dawn. If Judith is a knave, then it is not the case that: Keith is immediately before Dawn. If Juanita is a knight, then Dawn is immediately before Grace. If Juanita is a knave, then it is not the case that: Dawn is immediately before Grace. Which of the following is correct?\nA) [must be true] Judith is in slot 5\nB) [could be false] Judith is in slot 2",
  "active_domains": [
   "knights_and_knaves",
   "ordering"
  ]
 },
 {
  "problem_text": "There are 5 positions numbered 1 through 5. Zara, Annette, Raymond, Patricia, and Jean are assigned to these positions. Raymond is in slot 2. Patricia is in slot 3. Patricia is immediately before Annette. Annette and Patricia are adjacent. Which of the following is correct?\nA) [must be true] Annette is in slot 4\nB) [must be true] Annette is in slot 1\nC) [could be true] Annette is in slot 3\nD) [could be true] Annette is in slot 2",
  "active_domains": [
   "ordering"
  ]
 },
 {
  "problem_text": "There are 5 positions (numbered 1 through 5). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Liam, Pamela, Troy, Cyrus, and Marianna are the people in this scenario. Troy is before Pamela. Liam and Marianna are in different groups. Pamela and Cyrus are in the same group. It is not the case that: Pamela and Troy are in different groups. If exactly 2 of [Liam, Troy, Marianna] are in group 2, then Cyrus is before Troy. If Pamela is a truth-teller (knight), then Liam and Troy are in the same group. If Pamela is a deceiver (knave), then it is not the case that: Liam and Troy are in the same group. If Liam is a truth-teller (knight), then Troy is a deceiver (knave). If Liam is a deceiver (knave), then it is not the case that: Troy is a deceiver (knave). Which of the following is correct?\nA) [could be true] Pamela is in group 2\nB) [must be true] Pamela is in group 1",
  "active_domains": [
   "grouping",
   "knights_and_knaves",
   "ordering"
  ]
 },
 {
  "problem_text": "There are 2 positions (numbered 1 through 2). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Pamela is in slot 1. Pamela is in group 2. Exactly 1 of [Pamela, Evelyn] are in group 1. Exactly 1 of [Pamela, Evelyn] are in group 2. If Evelyn is in group 1, then Pamela is in slot 1. If Pamela is immediately before Evelyn, then exactly 1 of [Pamela, Evelyn] are in group 2. If Evelyn is a truth-teller (knight), then Pamela is in slot 2. If Evelyn is a deceiver (knave), then it is not the case that: Pamela is in slot 2. If Evelyn is a truth-teller (knight), then Evelyn is before Pamela. If Evelyn is a deceiver (knave), then it is not the case that: Evelyn is before Pamela. Which of the following is correct?\nA) [could be false] Pamela is a deceiver (knave)\nB) [must be true] Pamela is a truth-teller (knight)",
  "active_domains": [
   "knights_and_knaves",
   "ordering",
   "grouping"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Bonnie and Mars are in different groups. Rosetta and Sharon are in different groups. Exactly 3 of [Bonnie, Carlene, Mars, Rosetta, Sharon] are in group 1. Exactly 2 of [Mars, Rosetta, Sharon] are in group 1. If Sharon is a truth-teller (knight), then exactly 2 of [Bonnie, Carlene, Mars] are in group 1. If Sharon is a deceiver (knave), then it is not the case that: exactly 2 of [Bonnie, Carlene, Mars] are in group 1. If Sharon is a truth-teller (knight), then exactly 1 of [Bonnie, Rosetta, Sharon] are in group 1. If Sharon is a deceiver (knave), then it is not the case that: exactly 1 of [Bonnie, Rosetta, Sharon] are in group 1. Which of the following is correct?\nA) [must be true] Bonnie is a truth-teller (knight)\nB) [could be true] Bonnie is a deceiver (knave)",
  "active_domains": [
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 3 positions (numbered 1 through 3). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Ivan is before Morgan. Eddie is in group 1. If Morgan and Ivan are in the same group, then Morgan and Eddie are adjacent. If Morgan and Eddie are adjacent, then Morgan is in group 2. If Morgan is a truth-teller (knight), then Eddie and Ivan are in different groups. If Morgan is a deceiver (knave), then it is not the case that: Eddie and Ivan are in different groups. If Morgan is a truth-teller (knight), then Ivan is in slot 1. If Morgan is a deceiver (knave), then it is not the case that: Ivan is in slot 1. Given that Ivan is immediately before Morgan, which of the following is correct?\nA) [could be true] Ivan is in slot 1\nB) [must be true] Ivan is in slot 2\nC) [could be true] Ivan is in slot 3",
  "active_domains": [
   "grouping",
   "ordering",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 6 positions (numbered 1 through 6). Every person is either a knight (always tells the truth) or a knave (always lies). Olga is before Victor. Victor and Orla are adjacent. Janet is a truth-teller (knight). If Orla is in slot 5, then Olga is before Orla. If Michael is a truth-teller (knight), then Sherry is in slot 3. If Orla is a truth-teller (knight), then Sherry is in slot 3. If Orla is a deceiver (knave), then it is not the case that: Sherry is in slot 3. If Janet is a truth-teller (knight), then Sherry and Victor are adjacent. If Janet is a deceiver (knave), then it is not the case that: Sherry and Victor are adjacent. Given that if Olga is a truth-teller (knight), then Janet is before Michael, which of the following is correct?\nA) [could be false] Janet is in slot 2\nB) [must be false] Janet is in slot 5",
  "active_domains": [
   "knights_and_knaves",
   "ordering"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). Billy is a deceiver (knave). Linda is a truth-teller (knight). Dawn is a truth-teller (knight). If Dawn is a truth-teller (knight), then Billy is a deceiver (knave). If Dawn is a deceiver (knave), then it is not the case that: Billy is a deceiver (knave). If Linda is a truth-teller (knight), then Billy is a deceiver (knave). If Linda is a deceiver (knave), then it is not the case that: Billy is a deceiver (knave). Which of the following is correct?\nA) [must be true] Dawn is a truth-teller (knight)\nB) [could be true] Dawn is a deceiver (knave)",
  "active_domains": [
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Luna is a knave. Luna and Veronica are in different groups. Veronica is in group 2. If Luna is a knave, then Veronica is in group 2. If Orla is a knight, then Luna is a knave. If Orla is a knave, then it is not the case that: Luna is a knave. If Veronica is a knight, then Luna is in group 1. If Veronica is a knave, then it is not the case that: Luna is in group 1. Given that if Veronica is a knight, then Luna and Veronica are in different groups, which of the following is correct?\nA) [could be false] Orla is in group 1\nB) [must be false] Orla is in group 2",
  "active_domains": [
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). Morgan is a truth-teller (knight). Sage is a deceiver (knave). Eddie is a truth-teller (knight). If Sage is a truth-teller (knight), then Eddie is a deceiver (knave). If Sage is a deceiver (knave), then it is not the case that: Eddie is a deceiver (knave). If Eddie is a truth-teller (knight), then Sage is a deceiver (knave). If Eddie is a deceiver (knave), then it is not the case that: Sage is a deceiver (knave). Which of the following is correct?\nA) [could be true] Morgan is a deceiver (knave)\nB) [must be true] Morgan is a truth-teller (knight)",
  "active_domains": [
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 6 positions numbered 1 through 6. There are 3 groups. Rosetta, Edgar, Sharon, Sage, Samuel, and Anthony are assigned to these positions and groups. Sharon and Sage are in different groups. If Edgar is in group 1, then Edgar is before Sage. If exactly 1 of [Edgar, Sage, Samuel, Anthony] are in group 1, then Sharon is before Anthony. If Anthony is immediately before Sage, then exactly 2 of [Rosetta, Edgar, Sharon, Sage, Samuel, Anthony] are in group 1. Which of the following is correct?\nA) [must be true] Edgar is in slot 1\nB) [could be true] Edgar is in slot 4",
  "active_domains": [
   "ordering",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 6 positions (numbered 1 through 6). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Exactly 1 of [Ray, Reed, Mary] are in group 1. Exactly 3 of [Ray, Reed, Mary, Tommie] are in group 2. Exactly 3 of [Reed, Mary, Omar, Donald] are in group 2. If exactly 3 of [Mary, Omar, Tommie, Donald] are in group 2, then exactly 2 of [Ray, Omar, Donald] are in group 1. If Tommie is before Donald, then exactly 2 of [Reed, Mary, Omar] are in group 2. If Ray is before Reed, then exactly 2 of [Ray, Omar, Tommie, Donald] are in group 1. If Reed is a truth-teller (knight), then Ray is in slot 4. If Reed is a deceiver (knave), then it is not the case that: Ray is in slot 4. If Mary is a truth-teller (knight), then exactly 2 of [Ray, Mary, Omar, Tommie] are in group 2. If Mary is a deceiver (knave), then it is not the case that: exactly 2 of [Ray, Mary, Omar, Tommie] are in group 2. Given that exactly 1 of [Reed, Mary, Omar, Tommie] are in group 1, which of the following is correct?\nA) [could be false] Reed is a deceiver (knave)\nB) [must be false] Reed is a truth-teller (knight)",
  "active_domains": [
   "knights_and_knaves",
   "grouping",
   "ordering"
  ]
 },
 {
  "problem_text": "There are 6 positions (numbered 1 through 6). Every person is either a knight (always tells the truth) or a knave (always lies). The people are Shayne, Randolph, Jacquelyn, Priya, Salvatore, and Lisa. If Jacquelyn is before Lisa, then Lisa is in slot 5. If Salvatore is before Shayne, then Lisa is a truth-teller (knight). If Jacquelyn is a truth-teller (knight), then Shayne is before Lisa. If Jacquelyn is a truth-teller (knight), then Randolph is a deceiver (knave). If Jacquelyn is a deceiver (knave), then it is not the case that: Randolph is a deceiver (knave). If Randolph is a truth-teller (knight), then Lisa is before Jacquelyn. If Randolph is a deceiver (knave), then it is not the case that: Lisa is before Jacquelyn. Given that Salvatore is before Priya, which of the following is correct?\nA) [must be true] Salvatore is a deceiver (knave)\nB) [could be true] Salvatore is a truth-teller (knight)",
  "active_domains": [
   "knights_and_knaves",
   "ordering"
  ]
 },
 {
  "problem_text": "There are 2 positions numbered 1 through 2. Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Omar is before Heidi. Omar and Heidi are in different groups. Heidi is in group 1. If Omar is in slot 1, then Heidi is a truth-teller (knight). If Omar is a truth-teller (knight), then Omar is before Heidi. If Omar is a deceiver (knave), then it is not the case that: Omar is before Heidi. If Heidi is a truth-teller (knight), then Omar is before Heidi. If Heidi is a deceiver (knave), then it is not the case that: Omar is before Heidi. Given that if Omar and Heidi are adjacent, then Omar is in group 2, which of the following is correct?\nA) [could be true] Heidi is in slot 1\nB) [must be true] Heidi is in slot 2",
  "active_domains": [
   "grouping",
   "ordering",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "Amelia, Erica, Gary, Samatha, Yolanda, and Luna stand in six positions numbered 1 through 6. Each person is either a knight, who always tells the truth, or a knave, who always lies. There are three groups.\n\nexactly 1 of [Amelia, Gary, Samatha] are in group 3.\nexactly 1 of [Amelia, Yolanda, Luna] are in group 3.\nexactly 2 of [Erica, Gary, Samatha, Yolanda] are in group 3.\nIf Samatha and Yolanda are in different groups, then Gary is a truth-teller (knight).\nIf Gary is immediately before Yolanda, then exactly 2 of [Amelia, Samatha, Luna] are in group 3.\nIf Luna is in slot 4, then exactly 1 of [Gary, Yolanda, Luna] are in group 2.\nIf Amelia is a truth-teller (knight), then exactly 2 of [Amelia, Erica, Gary, Luna] are in group 3.\nIf Amelia is a deceiver (knave), then it is not the case that: exactly 2 of [Amelia, Erica, Gary, Luna] are in group 3.\nIf Luna is a truth-teller (knight), then Luna is in slot 2.\nIf Luna is a deceiver (knave), then it is not the case that: Luna is in slot 2.\n\nGiven that exactly 2 of [Samatha, Yolanda, Luna] are in group 3.\n\nWhich of the following is correct?\nA) [could be true] Samatha is in slot 1\nB) [must be false] Samatha is in slot 4\nC) [must be true] Samatha is in slot 5",
  "active_domains": [
   "knights_and_knaves",
   "ordering",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 6 positions (numbered 1 through 6). Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Dara is before Raphael. Exactly 1 of [Juanita, Raphael, Kenneth] are in group 2. Exactly 1 of [Blanca, Juanita, Nina, Kenneth] are in group 2. If Blanca and Dara are in different groups, then Raphael is immediately before Nina. If Dara is in slot 1, then exactly 1 of [Blanca, Juanita, Raphael, Kenneth] are in group 1. If Blanca is in slot 2, then exactly 2 of [Blanca, Juanita, Raphael] are in group 3. If Juanita is a truth-teller (knight), then Blanca and Kenneth are in different groups. If Juanita is a deceiver (knave), then it is not the case that: Blanca and Kenneth are in different groups. If Kenneth is a truth-teller (knight), then exactly 2 of [Dara, Juanita, Raphael, Kenneth] are in group 2. If Kenneth is a deceiver (knave), then it is not the case that: exactly 2 of [Dara, Juanita, Raphael, Kenneth] are in group 2. Given that if exactly 2 of [Blanca, Dara, Raphael, Kenneth] are in group 2, then Raphael is immediately before Nina, which of the following is correct?\nA) [must be true] Juanita is a deceiver (knave)\nB) [could be false] Juanita is a truth-teller (knight)",
  "active_domains": [
   "grouping",
   "knights_and_knaves",
   "ordering"
  ]
 },
 {
  "problem_text": "There are 5 positions (numbered 1 through 5). Every person is either a knight (always tells the truth) or a knave (always lies). Xander is before Scott. Knox is before Lynda. Luna is before Lynda. If Knox is immediately before Luna, then Lynda is a truth-teller (knight). If Xander is immediately before Scott, then Lynda is a truth-teller (knight). If Scott is a truth-teller (knight), then Scott is immediately before Xander. If Scott is a deceiver (knave), then it is not the case that: Scott is immediately before Xander. If Luna is a truth-teller (knight), then Xander and Knox are adjacent. If Luna is a deceiver (knave), then it is not the case that: Xander and Knox are adjacent. Given that Scott is a deceiver (knave), which of the following is correct?\nA) [must be true] Knox is a deceiver (knave)\nB) [could be true] Knox is a truth-teller (knight)",
  "active_domains": [
   "ordering",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). Rick is a truth-teller (knight). Maple is a deceiver (knave). If Rick is a truth-teller (knight), then Maple is a deceiver (knave). If Rick is a deceiver (knave), then it is not the case that: Maple is a deceiver (knave). If Maple is a truth-teller (knight), then Rick is a deceiver (knave). If Maple is a deceiver (knave), then it is not the case that: Rick is a deceiver (knave). Which of the following is correct?\nA) [could be true] Rick is a truth-teller (knight)\nB) [could be true] Rick is a deceiver (knave)",
  "active_domains": [
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 3 positions numbered 1 through 3. Sandra, Leone, and Heidi are in these positions. Leone is before Sandra. Heidi is before Sandra. If Leone is in slot 1, then Sandra and Heidi are adjacent. Given that Leone is before Heidi, which of the following is correct?\nA) [must be true] Sandra is in slot 1\nB) [could be true] Sandra is in slot 3\nC) [must be true] Sandra is in slot 2",
  "active_domains": [
   "ordering"
  ]
 },
 {
  "problem_text": "There are 4 positions (numbered 1 through 4). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Delbert is before Robert. Delbert and Nova are in different groups. If exactly 1 of [Grace, Delbert, Nova] are in group 1, then Delbert is before Grace. If exactly 2 of [Delbert, Robert, Nova] are in group 2, then Grace is before Robert. If Nova is in slot 2, then Grace and Nova are in the same group. If Grace is a truth-teller (knight), then Delbert and Robert are in different groups. If Grace is a deceiver (knave), then it is not the case that: Delbert and Robert are in different groups. If Robert is a truth-teller (knight), then exactly 1 of [Delbert, Robert, Nova] are in group 1. If Robert is a deceiver (knave), then it is not the case that: exactly 1 of [Delbert, Robert, Nova] are in group 1. Which of the following is correct?\nA) [must be true] Grace is in slot 2\nB) [must be true] Grace is in slot 4\nC) [could be false] Grace is in slot 3",
  "active_domains": [
   "knights_and_knaves",
   "grouping",
   "ordering"
  ]
 },
 {
  "problem_text": "There are 5 positions (numbered 1 through 5). Every person is either a knight (always tells the truth) or a knave (always lies). Miguel, Daniel, Priya, Linda, and Zara are to be arranged in these five positions. Daniel is in slot 2. Linda is before Priya. If Zara is a deceiver (knave), then Miguel is immediately before Linda. If Zara is a deceiver (knave), then Daniel is before Miguel. If Miguel is before Priya, then Zara is a deceiver (knave). If Linda is a truth-teller (knight), then Zara is before Miguel. If Linda is a deceiver (knave), then it is not the case that: Zara is before Miguel. If Miguel is a truth-teller (knight), then Priya and Zara are adjacent. If Miguel is a deceiver (knave), then it is not the case that: Priya and Zara are adjacent. Which of the following is correct? A) [could be false] Daniel is a truth-teller (knight) B) [must be false] Daniel is a deceiver (knave)",
  "active_domains": [
   "knights_and_knaves",
   "ordering"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Nina is a truth-teller (knight). Nina is in group 2. exactly 1 of [Nina, Marian] are in group 1. If exactly 1 of [Nina, Marian] are in group 2, then Nina is a truth-teller (knight). If Marian is in group 1, then Nina is a truth-teller (knight). If Nina is a truth-teller (knight), then Marian is a deceiver (knave). If Nina is a deceiver (knave), then it is not the case that: Marian is a deceiver (knave). If Nina is a truth-teller (knight), then exactly 1 of [Nina, Marian] are in group 2. If Nina is a deceiver (knave), then it is not the case that: exactly 1 of [Nina, Marian] are in group 2. Given that if Nina is in group 2, then Marian is a deceiver (knave), which of the following is correct?\nA) [must be true] Nina is a deceiver (knave)\nB) [could be true] Nina is a truth-teller (knight)",
  "active_domains": [
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 2 groups. Finn, Iris, Mars, Luisa, and Marianna are assigned to these groups. Mars and Luisa are in the same group. Finn is in group 1. Marianna is in group 2. Exactly 1 of Finn, Mars, Luisa, Marianna are in group 2. Exactly 1 of Iris, Mars, Luisa, Marianna are in group 2. Which of the following is correct?\nA) [could be true] Iris is in group 1\nB) [must be true] Iris is in group 2",
  "active_domains": [
   "grouping"
  ]
 },
 {
  "problem_text": "There are 3 positions numbered 1 through 3. Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. George, Tara, and Vega are the people involved. George is before Vega. Tara is in group 1. Exactly 1 of [George, Tara, Vega] are in group 3. Exactly 1 of [George, Tara, Vega] are in group 1. If Vega is in slot 3, then exactly 1 of [George, Tara, Vega] are in group 1. If Vega is a truth-teller (knight), then exactly 1 of [George, Tara, Vega] are in group 2. If Vega is a deceiver (knave), then it is not the case that: exactly 1 of [George, Tara, Vega] are in group 2. If Vega is a truth-teller (knight), then exactly 1 of [George, Tara, Vega] are in group 3. If Vega is a deceiver (knave), then it is not the case that: exactly 1 of [George, Tara, Vega] are in group 3. Which of the following is correct?\nA) [could be false] Vega is in slot 3\nB) [must be true] Vega is in slot 1\nC) [must be false] Vega is in slot 2",
  "active_domains": [
   "ordering",
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 3 positions (numbered 1 through 3). Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Evan and Mars are in different groups. Mars is in group 2. If Hollis and Mars are adjacent, then Evan is in group 3. If Evan is a truth-teller (knight), then Evan is in group 1. If Evan is a deceiver (knave), then it is not the case that: Evan is in group 1. If Hollis is a truth-teller (knight), then Hollis and Evan are in different groups. If Hollis is a deceiver (knave), then it is not the case that: Hollis and Evan are in different groups. Given that if Hollis and Mars are in different groups, then Mars is in slot 3, which of the following is correct?\nA) [could be false] Mars is in slot 3\nB) [must be true] Mars is in slot 1\nC) [must be true] Mars is in slot 2",
  "active_domains": [
   "knights_and_knaves",
   "ordering",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 5 positions (numbered 1 through 5). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Alfonso is before Liam. Brian and Andy are adjacent. exactly 1 of [Andy, Alfonso, Liam, Joseph] are in group 2. If exactly 2 of [Brian, Andy, Liam] are in group 1, then Andy is before Joseph. If Andy is in slot 1, then Alfonso is a truth-teller (knight). If Alfonso is a truth-teller (knight), then Brian and Andy are in the same group. If Alfonso is a deceiver (knave), then it is not the case that: Brian and Andy are in the same group. If Joseph is a truth-teller (knight), then Andy is in slot 2. If Joseph is a deceiver (knave), then it is not the case that: Andy is in slot 2. Which of the following is correct?\nA) [could be false] Brian is a truth-teller (knight)\nB) [must be true] Brian is a deceiver (knave)",
  "active_domains": [
   "ordering",
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 6 positions numbered 1 through 6. Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Bernard, Norman, Nova, Troy, Lois, and Colleen are the individuals. Exactly 1 of [Bernard, Norman, Nova, Troy, Lois, Colleen] are in group 2. Exactly 2 of [Norman, Lois, Colleen] are in group 3. Exactly 2 of [Bernard, Nova, Troy, Lois] are in group 3. Exactly 3 of [Bernard, Nova, Lois, Colleen] are in group 3. Exactly 1 of [Norman, Troy, Lois, Colleen] are in group 1. If Lois is a truth-teller (knight), then Norman is in group 3. If Lois is a deceiver (knave), then it is not the case that: Norman is in group 3. If Bernard is a truth-teller (knight), then Troy is in slot 2. If Bernard is a deceiver (knave), then it is not the case that: Troy is in slot 2. Given that Lois is before Norman, which of the following is correct?\nA) [could be false] Nova is in group 3\nB) [must be true] Nova is in group 1\nC) [could be true] Nova is in group 2",
  "active_domains": [
   "knights_and_knaves",
   "grouping",
   "ordering"
  ]
 },
 {
  "problem_text": "There are 2 groups. Sandra, Wendy, and Josefina are assigned to these groups. Sandra and Josefina are in the same group. Wendy and Josefina are in different groups. Sandra is in group 2. Josefina is in group 2. exactly 1 of [Sandra, Wendy, Josefina] are in group 1. Given that exactly 2 of [Sandra, Wendy, Josefina] are in group 2, which of the following is correct?\nA) [must be true] Sandra is in group 2\nB) [could be true] Sandra is in group 1",
  "active_domains": [
   "grouping"
  ]
 },
 {
  "problem_text": "There are 2 positions (numbered 1 through 2). Every person is either a knight (always tells the truth) or a knave (always lies). Joan is before Regina. If Regina is a truth-teller (knight), then Joan is before Regina. If Regina is a truth-teller (knight), then Joan is immediately before Regina. If Regina is a truth-teller (knight), then Joan is in slot 1. If Regina is a deceiver (knave), then it is not the case that: Joan is in slot 1. If Joan is a truth-teller (knight), then Regina is before Joan. If Joan is a deceiver (knave), then it is not the case that: Regina is before Joan. Which of the following is correct?\nA) [could be true] Joan is a deceiver (knave)\nB) [could be true] Joan is a truth-teller (knight)",
  "active_domains": [
   "ordering",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies).\nMajor, Lisa, and Dave are in play.\nMajor is a truth-teller (knight).\nLisa is a deceiver (knave).\nDave is a deceiver (knave).\nIf Major is a truth-teller (knight), then Dave is a deceiver (knave).\nIf Major is a deceiver (knave), then it is not the case that Dave is a deceiver (knave).\nIf Lisa is a truth-teller (knight), then Major is a deceiver (knave).\nIf Lisa is a deceiver (knave), then it is not the case that Major is a deceiver (knave).\nWhich of the following is correct?\nA) [must be true] Major is a truth-teller (knight)\nB) [must be true] Major is a deceiver (knave)",
  "active_domains": [
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Exactly 2 of [Mars, Jeff, Morgan] are in group 2. Exactly 1 of [Jeff, Morgan, Lena] are in group 3. Exactly 1 of [Mars, Morgan, Priya, Lena] are in group 2. If exactly 2 of [Mars, Jeff, Morgan, Priya] are in group 2, then Lena is a truth-teller (knight). If Jeff is a truth-teller (knight), then exactly 1 of [Jeff, Morgan, Priya, Lena] are in group 1. If Jeff is a deceiver (knave), then it is not the case that: exactly 1 of [Jeff, Morgan, Priya, Lena] are in group 1. If Lena is a truth-teller (knight), then exactly 2 of [Mars, Morgan, Priya] are in group 3. If Lena is a deceiver (knave), then it is not the case that: exactly 2 of [Mars, Morgan, Priya] are in group 3. Which of the following is correct?\nA) [could be false] Mars is a truth-teller (knight)\nB) [must be true] Mars is a deceiver (knave)",
  "active_domains": [
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 3 groups. Eric, Betty, and Gwen are each assigned to one of these groups. Eric and Betty are in different groups. Eric and Gwen are in different groups. Eric is in group 1. Exactly 1 of Eric, Betty, and Gwen are in group 2. Exactly 1 of Eric, Betty, and Gwen are in group 3. Exactly 1 of Eric, Betty, and Gwen are in group 2. Given that exactly 1 of Eric, Betty, and Gwen are in group 1, which of the following is correct?\nA) [could be true] Gwen is in group 1\nB) [could be false] Gwen is in group 2",
  "active_domains": [
   "grouping"
  ]
 },
 {
  "problem_text": "There are 2 positions numbered 1 through 2. Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Maurice is immediately before Martin. Martin and Maurice are in different groups. If exactly 1 of [Martin, Maurice] are in group 1, then Martin is a truth-teller (knight). If Maurice is in slot 1, then exactly 1 of [Martin, Maurice] are in group 2. If Martin is a truth-teller (knight), then exactly 1 of [Martin, Maurice] are in group 2. If Martin is a deceiver (knave), then it is not the case that: exactly 1 of [Martin, Maurice] are in group 2. If Maurice is a truth-teller (knight), then Maurice is in slot 1. If Maurice is a deceiver (knave), then it is not the case that: Maurice is in slot 1. Which of the following is correct?\nA) [must be false] Martin is in group 1\nB) [could be false] Martin is in group 2",
  "active_domains": [
   "ordering",
   "knights_and_knaves",
   "grouping"
  ]
 },
 {
  "problem_text": "There are 3 positions (numbered 1 through 3). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Carol, Knox, and Victor are the people in the positions. Victor is before Knox. Victor is immediately before Knox. Carol and Knox are adjacent. If exactly 2 of [Carol, Knox, Victor] are in group 1, then Carol and Knox are adjacent. If Knox is a truth-teller (knight), then Knox is immediately before Victor. If Knox is a deceiver (knave), then it is not the case that: Knox is immediately before Victor. If Victor is a truth-teller (knight), then Knox is in slot 2. If Victor is a deceiver (knave), then it is not the case that: Knox is in slot 2. Which of the following is correct?\nA) [must be true] Knox is in group 1\nB) [could be true] Knox is in group 2",
  "active_domains": [
   "grouping",
   "knights_and_knaves",
   "ordering"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). Arthur is a deceiver (knave). Bernard is a deceiver (knave). Tara is a deceiver (knave). Sergio is a deceiver (knave). Heidi is a deceiver (knave). If Heidi is a truth-teller (knight), then Tara is a truth-teller (knight). If Heidi is a deceiver (knave), then it is not the case that: Tara is a truth-teller (knight). If Bernard is a truth-teller (knight), then Heidi is a truth-teller (knight). If Bernard is a deceiver (knave), then it is not the case that: Heidi is a truth-teller (knight). Which of the following is correct?\nA) [must be true] Arthur is a deceiver (knave)\nB) [could be true] Arthur is a truth-teller (knight)",
  "active_domains": [
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 5 positions (numbered 1 through 5). Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Don, Lauren, Vega, Gordon, and Benjamin are the people involved. Benjamin is before Don. Benjamin is before Lauren. Benjamin is before Gordon. Lauren and Vega are adjacent. exactly 1 of [Don, Lauren, Vega] are in group 3. exactly 1 of [Lauren, Vega, Gordon] are in group 2. If Vega is a truth-teller (knight), then Lauren is in slot 3. If Vega is a deceiver (knave), then it is not the case that: Lauren is in slot 3. If Benjamin is a truth-teller (knight), then exactly 1 of [Lauren, Vega, Gordon] are in group 3. If Benjamin is a deceiver (knave), then it is not the case that: exactly 1 of [Lauren, Vega, Gordon] are in group 3. Given that Don is in slot 5, which of the following is correct?\nA) [must be false] Don is in group 2\nB) [must be true] Don is in group 3\nC) [could be true] Don is in group 1",
  "active_domains": [
   "knights_and_knaves",
   "grouping",
   "ordering"
  ]
 },
 {
  "problem_text": "There are 5 positions (numbered 1 through 5).\nDaniel, Willie, Pearline, David, and Samatha are the entities.\nPearline is in slot 4.\nDavid is in slot 1.\nDavid is before Pearline.\nWillie is immediately before Daniel.\nDaniel and Pearline are adjacent.\nit is not the case that: David is in slot 3.\nGiven that Daniel and Willie are adjacent, which of the following is correct?\nA) [must be true] David is in slot 2\nB) [must be true] David is in slot 5\nC) [could be true] David is in slot 1\nD) [must be true] David is in slot 3",
  "active_domains": [
   "ordering"
  ]
 },
 {
  "problem_text": "There are 6 positions (numbered 1 through 6). Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Andrew, Chung, Heather, Luna, Allen, Tina are the people. Chung is immediately before Andrew. Heather is a truth-teller (knight). Luna is a truth-teller (knight). Allen is a deceiver (knave). Tina is a truth-teller (knight). exactly 1 of [Chung, Heather, Luna] are in group 2. exactly 1 of [Luna, Allen, Tina] are in group 3. If exactly 2 of [Chung, Heather, Tina] are in group 1, then exactly 2 of [Andrew, Chung, Tina] are in group 1. If exactly 1 of [Andrew, Luna, Allen] are in group 1, then Allen is before Tina. If Luna and Tina are in different groups, then Chung is before Allen. If exactly 1 of [Andrew, Allen, Tina] are in group 3, then Andrew is a deceiver (knave). If Tina is before Heather, then exactly 1 of [Chung, Heather, Allen, Tina] are in group 2. If Andrew is before Tina, then Chung is a truth-teller (knight). If Chung is immediately before Andrew, then exactly 2 of [Heather, Allen, Tina] are in group 1. If Andrew and Chung are adjacent, then exactly 2 of [Andrew, Luna, Allen] are in group 3. If Chung is a truth-teller (knight), then exactly 2 of [Andrew, Chung, Luna, Tina] are in group 1. If exactly 1 of [Andrew, Allen, Tina] are in group 3, then Luna is in slot 1. If Andrew and Chung are in different groups, then Tina is immediately before Heather. If Allen is a truth-teller (knight), then Chung is immediately before Allen. If Allen is a deceiver (knave), then it is not the case that: Chung is immediately before Allen. If Tina is a truth-teller (knight), then exactly 2 of [Andrew, Heather, Allen, Tina] are in group 1. If Tina is a deceiver (knave), then it is not the case that: exactly 2 of [Andrew, Heather, Allen, Tina] are in group 1. Which of the following is correct?\nA) [must be true] Heather is in slot 6\nB) [must be true] Heather is in slot 5\nC) [must be true] Heather is in slot 2",
  "active_domains": [
   "grouping",
   "ordering",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 2 positions (numbered 1 through 2). Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Naomi is in slot 2. Naomi is a deceiver (knave). Sidney is a deceiver (knave). Naomi is in group 1. If Naomi and Sidney are adjacent, then Naomi and Sidney are in different groups. If Naomi is a truth-teller (knight), then Naomi is immediately before Sidney. If Naomi is a deceiver (knave), then it is not the case that: Naomi is immediately before Sidney. If Sidney is a truth-teller (knight), then Naomi is immediately before Sidney. If Sidney is a deceiver (knave), then it is not the case that: Naomi is immediately before Sidney. Which of the following is correct?\nA) [must be true] Naomi is a deceiver (knave)\nB) [must be true] Naomi is a truth-teller (knight)",
  "active_domains": [
   "knights_and_knaves",
   "grouping",
   "ordering"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Juno is in group 1. If exactly 2 of [Brandi, Juno, Finn] are in group 1, then Juno is a truth-teller (knight). If exactly 2 of [William, Juno, Finn] are in group 2, then Juno is a truth-teller (knight). If Brandi is a truth-teller (knight), then exactly 2 of [Brandi, William, Finn] are in group 2. If Brandi is a deceiver (knave), then it is not the case that: exactly 2 of [Brandi, William, Finn] are in group 2. If Juno is a truth-teller (knight), then William is a deceiver (knave). If Juno is a deceiver (knave), then it is not the case that: William is a deceiver (knave). Which of the following is correct?\nA) [could be true] Brandi is a truth-teller (knight)\nB) [must be false] Brandi is a deceiver (knave)",
  "active_domains": [
   "grouping",
   "knights_and_knaves"
  ]
 },
 {
  "problem_text": "There are 2 positions numbered 1 through 2. Every person is either a knight (always tells the truth) or a knave (always lies). There are 2 groups. Theo is before William. It is not the case that: William is a truth-teller (knight). If exactly 1 of [William, Theo] are in group 2, then Theo is before William. If William is a deceiver (knave), then William and Theo are adjacent. If Theo is a truth-teller (knight), then exactly 1 of [William, Theo] are in group 2. If Theo is a deceiver (knave), then it is not the case that: exactly 1 of [William, Theo] are in group 2. If William is a truth-teller (knight), then William is before Theo. If William is a deceiver (knave), then it is not the case that: William is before Theo. Given that: William and Theo are in different groups, which of the following is correct?\nA) [could be true] Theo is in slot 1\nB) [must be true] Theo is in slot 2",
  "active_domains": [
   "grouping",
   "knights_and_knaves",
   "ordering"
  ]
 },
 {
  "problem_text": "There are 6 positions (numbered 1 through 6). There are 3 groups. Exactly 2 of [Ivan, Ronald, Donna] are in group 3. Exactly 1 of [Frank, Ivan, Vernon, Donna] are in group 2. If exactly 3 of [Ivan, Joseph, Vernon, Donna] are in group 3, then exactly 2 of [Frank, Ronald, Donna] are in group 3. If exactly 1 of [Frank, Ivan, Joseph, Ronald, Vernon, Donna] are in group 2, then Frank is immediately before Ronald. If Joseph and Donna are adjacent, then exactly 1 of [Ivan, Joseph, Ronald] are in group 2. Given that Ivan is in slot 3, which of the following is correct?\nA) [must be false] Vernon is in slot 5\nB) [must be true] Vernon is in slot 1\nC) [must be true] Vernon is in slot 2\nD) [could be false] Vernon is in slot 6",
  "active_domains": [
   "ordering",
   "grouping"
  ]
 },
 {
  "problem_text": "Every person is either a knight (always tells the truth) or a knave (always lies). There are 3 groups. Exactly 1 of [Cynthia, Frank, Mary] are in group 3. Exactly 1 of [Leonard, Frank, Mary, Craig] are in group 2. If exactly 1 of [Leonard, Frank, Mary] are in group 2, then Frank is a truth-teller (knight). If Yara is a truth-teller (knight), then Leonard is a deceiver (knave). If Yara is a deceiver (knave), then it is not the case that: Leonard is a deceiver (knave). If Frank is a truth-teller (knight), then exactly 3 of [Leonard, Yara, Mary, Craig] are in group 1. If Frank is a deceiver (knave), then it is not the case that: exactly 3 of [Leonard, Yara, Mary, Craig] are in group 1. Given that: if Cynthia is a truth-teller (knight), then exactly 1 of [Frank, Yara, Mary] are in group 1, which of the following is correct?\nA) [must be true] Cynthia is a deceiver (knave)\nB) [could be false] Cynthia is a truth-teller (knight)",
  "active_domains": [
   "grouping",
   "knights_and_knaves"
  ]
 }
]
print(len(ROWS), 'rows')


In [ ]:
# Same logic as ask_llm / classify_domains in pipeline.py
def new_usage():
    return {"in_tok": 0, "out_tok": 0}

def ask_llm(prompt, system, model, usage=None, timeout=300):
    body = {"model": model, "prompt": prompt, "system": system, "stream": False, "think": False}
    r = requests.post("http://localhost:11434/api/generate", json=body, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    if usage is not None:
        usage["in_tok"] += data.get("prompt_eval_count", 0)
        usage["out_tok"] += data.get("eval_count", 0)
    return data["response"].strip()

def classify_domains(problem_text, model, usage=None):
    response = ask_llm(problem_text, CLASSIFIER_SYSTEM, model, usage)
    domains = None
    for line in reversed(response.splitlines()):
        line = line.strip()
        if line.startswith("ACTIVE_DOMAINS:"):
            domains = [d.strip() for d in line.split(":", 1)[1].split(",") if d.strip()]
            break
    if domains is None:
        raise RuntimeError("Classifier did not emit ACTIVE_DOMAINS tag")
    unknown = [d for d in domains if d not in DOMAINS]
    if unknown:
        raise ValueError(f"Classifier returned unknown domains: {unknown}")
    return domains


In [ ]:
# Run every classifier over every row and compare against the gold domains
NUM_TEST_EXAMPLES = None  # None = all rows
rows = ROWS[:NUM_TEST_EXAMPLES] if NUM_TEST_EXAMPLES else ROWS
n = len(rows)
summary = []

for model in CLASSIFIERS:
    print(f"\n=== {model} ===")
    exact = covers = errors = 0
    total_ms = 0
    usage = new_usage()
    for i, row in enumerate(rows, 1):
        gold = set(row["active_domains"])
        print(f"  [{i}/{n}] ...", end="\r", flush=True)
        t0 = time.time()
        try:
            pred, err = classify_domains(row["problem_text"], model, usage), None
        except Exception as e:
            pred, err = None, str(e).splitlines()[0]
        total_ms += int((time.time() - t0) * 1000)
        if pred is None:
            errors += 1
            print(f"  [{i}/{n}] ERROR  gold={sorted(gold)}  {err}")
            continue
        pred = set(pred)
        exact += pred == gold
        covers += gold <= pred
        if pred != gold:
            print(f"  [{i}/{n}] WRONG  gold={sorted(gold)}  pred={sorted(pred)}")
    summary.append({"model": model, "exact": exact, "covers": covers, "errors": errors,
                    "avg_ms": total_ms / n, "avg_out": usage["out_tok"] / n})
    print(f"  exact {exact}/{n}  covers {covers}/{n}  errors {errors}")

print(f"\n{'=' * 72}\nCLASSIFIER SUMMARY  ({n} problems)\n{'=' * 72}")
w = max(len(s["model"]) for s in summary) + 2
print(f"  {'Model':<{w}}  {'Exact':<14}  {'Covers gold':<14}  {'Errors':<7}  {'Avg ms':<8}  Avg out tok")
for s in summary:
    ex = f"{s['exact']}/{n} ({s['exact'] / n * 100:.0f}%)"
    cv = f"{s['covers']}/{n} ({s['covers'] / n * 100:.0f}%)"
    print(f"  {s['model']:<{w}}  {ex:<14}  {cv:<14}  {s['errors']:<7}  {s['avg_ms']:<8.0f}  {s['avg_out']:.0f}")
